# exp021_hrv_metrics

**Track 3 — Biosignal Processing**

**Experiment:** `exp021`
**Description:** healthSpring Exp021 — HRV Metrics (RMSSD, pNN50)

Validates HRV metrics computed from Pan-Tompkins QRS detection:
- SDNN (standard deviation of NN intervals)
- RMSSD (root mean square of successive differences)
- pNN50 (percentage of successive RR intervals differing by > 50 ms)
- Mean RR interval, mean HR

Baseline is generated by running the Rust exp021_hrv_metrics binary
with --write-baseline (uses Rust synthetic ECG generator for consistency).

Provenance:
  Baseline source: Rust exp021_hrv_metrics --write-baseline
  Command:         python3 control/biosignal/exp021_hrv_metrics.py

**Source:** `control/biosignal/exp021_hrv_metrics.py`

## Paper Linkage

See `specs/PAPER_REVIEW_QUEUE.md` for the full paper → experiment mapping.
This notebook is the `.ipynb` pair of the `.py` control script for `exp021`.


In [ ]:
"""
healthSpring Exp021 — HRV Metrics (RMSSD, pNN50)

Validates HRV metrics computed from Pan-Tompkins QRS detection:
- SDNN (standard deviation of NN intervals)
- RMSSD (root mean square of successive differences)
- pNN50 (percentage of successive RR intervals differing by > 50 ms)
- Mean RR interval, mean HR

Baseline is generated by running the Rust exp021_hrv_metrics binary
with --write-baseline (uses Rust synthetic ECG generator for consistency).

Provenance:
  Baseline source: Rust exp021_hrv_metrics --write-baseline
  Command:         python3 control/biosignal/exp021_hrv_metrics.py
"""

import json
import os
import subprocess
import sys

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
REPO_ROOT = os.path.dirname(os.path.dirname(SCRIPT_DIR))
BASELINE_PATH = os.path.join(SCRIPT_DIR, "exp021_baseline.json")


def main():
    print("=" * 72)
    print("healthSpring Exp021: HRV Metrics (RMSSD, pNN50)")
    print("  Running Rust binary to generate baseline...")
    print("=" * 72)

    # Run Rust binary with --write-baseline to create exp021_baseline.json
    result = subprocess.run(
        ["cargo", "run", "--bin", "exp021_hrv_metrics", "--", "--write-baseline"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        print(result.stdout, file=sys.stderr)
        sys.exit(1)

    print(result.stdout)

    # Add provenance to the baseline
    if not os.path.exists(BASELINE_PATH):
        print(f"ERROR: Baseline not created at {BASELINE_PATH}", file=sys.stderr)
        sys.exit(1)

    with open(BASELINE_PATH, "r") as f:
        data = json.load(f)

    # Get git commit
    git_result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    git_commit = git_result.stdout.strip() if git_result.returncode == 0 else "unknown"

    data["_provenance"]["python"] = sys.version
    data["_provenance"]["git_commit"] = git_commit
    data["_provenance"]["command"] = "python3 control/biosignal/exp021_hrv_metrics.py"
    data["_provenance"]["script"] = "control/biosignal/exp021_hrv_metrics.py"

    with open(BASELINE_PATH, "w") as f:
        json.dump(data, f, indent=2)
        f.write("\n")

    print(f"\nBaseline written to {BASELINE_PATH} (with provenance)")
    return 0


if __name__ == "__main__":
    sys.exit(main())
